In [2]:
!pip install roboflow -q

from pathlib import Path
from dotenv import load_dotenv
import os

# Load .env from the project root
load_dotenv(Path('.env'))

RF_API_KEY = os.getenv('RF_API_KEY')
RF_WORKSPACE = os.getenv('RF_WORKSPACE')
RF_PROJECT = os.getenv('RF_PROJECT')
RF_VERSION = int(os.getenv('RF_VERSION', '1'))


from roboflow import Roboflow

rf = Roboflow(api_key=RF_API_KEY)
version = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION)
dataset = version.download('yolov8')

DATASET_DIR = Path(dataset.location)
print(f'Downloaded to: {DATASET_DIR}')

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to HairNet-Compliance-Detection-1 in yolov8:: 100%|██████████| 851/851 [00:01<00:00, 787.89it/s] 

Downloaded to: /content/HairNet-Compliance-Detection-1


# HairNet Compliance Detection

### MLflow Training

Each run varies:
- Model size (`yolov26n`, `yolov26m`, `yolo12n`, `yolo12m`)
- Epochs
- Image size
- Augmentation settings

At the end, MLflow lets us compare all runs and pick the winner.

## 0. Install Dependencies

In [3]:
!pip install ultralytics mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.6/88

## 1. Imports & Setup

In [4]:
import os
import sys
from pathlib import Path

import mlflow
import mlflow.artifacts
import pandas as pd
from ultralytics import YOLO


DATA_YAML = Path('/content/HairNet-Compliance-Detection-1/data.yaml')
assert DATA_YAML.exists(), f'data.yaml not found at {DATA_YAML}. Run the download cell first.'

# MLflow tracking

MLFLOW_TRACKING_URI = 'mlruns'
EXPERIMENT_NAME     = 'hairnet-compliance-detection'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'MLflow tracking URI : {mlflow.get_tracking_uri()}')
print(f'Experiment          : {EXPERIMENT_NAME}')
print(f'Data YAML           : {DATA_YAML.resolve()}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
2026/05/13 17:35:38 INFO mlflow.tracking.fluent: Experiment with name 'hairnet-compliance-detection' does not exist. Creating a new experiment.


MLflow tracking URI : mlruns
Experiment          : hairnet-compliance-detection
Data YAML           : /content/HairNet-Compliance-Detection-1/data.yaml


## 2. Define Experiment Configurations

Each dict is one MLflow run. Add or remove entries to control what gets trained.

#### [yolo AUG doc] https://docs.ultralytics.com/guides/yolo-data-augmentation

In [5]:
AUG_OFF = dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, translate=0.0,
               scale=0.0, fliplr=0.0, mosaic=0.0, erasing=0.0, auto_augment=None)

AUG_ON = dict(hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, translate=0.1, scale=0.5,
              fliplr=0.5, flipud=0.1, mosaic=1.0,
              erasing=0.4, degrees=10.0, auto_augment='randaugment')

RUNS = [
    # ── Baselines (no augmentation) ───────────────────────────────────
    {'run_name': 'yolov26n-baseline', 'model': 'yolo26n.pt', 'epochs': 30, 'batch': 16, **AUG_OFF},
    {'run_name': 'yolov26m-baseline', 'model': 'yolo26m.pt', 'epochs': 30, 'batch': 8,  **AUG_OFF},
    {'run_name': 'yolo12n-baseline',  'model': 'yolo12n.pt',  'epochs': 30, 'batch': 16, **AUG_OFF},
    {'run_name': 'yolo12m-baseline',  'model': 'yolo12m.pt',  'epochs': 30, 'batch': 8,  **AUG_OFF},

    # ── Augmented ─────────────────────────────────────────────────────
    {'run_name': 'yolov26n-augmented', 'model': 'yolo26n.pt', 'epochs': 50, 'batch': 16, **AUG_ON},
    {'run_name': 'yolov26m-augmented', 'model': 'yolo26m.pt', 'epochs': 50, 'batch': 8,  **AUG_ON},
    {'run_name': 'yolo12n-augmented',  'model': 'yolo12n.pt',  'epochs': 50, 'batch': 16, **AUG_ON},
    {'run_name': 'yolo12m-augmented',  'model': 'yolo12m.pt',  'epochs': 50, 'batch': 8,  **AUG_ON},
]

# shared params applied to every run
SHARED = dict(imgsz=640, optimizer='auto', patience=10)

print(f'{len(RUNS)} runs defined:\n')
print(f"  {'Run':<25} {'Model':<14} {'Epochs':>6} {'Batch':>6} {'Aug'}")
print('  ' + '-' * 60)
for r in RUNS:
    aug = 'ON' if r['mosaic'] > 0 else 'OFF'
    print(f"  {r['run_name']:<25} {r['model']:<14} {r['epochs']:>6} {r['batch']:>6}   {aug}")

8 runs defined:

  Run                       Model          Epochs  Batch Aug
  ------------------------------------------------------------
  yolov26n-baseline         yolo26n.pt         30     16   OFF
  yolov26m-baseline         yolo26m.pt         30      8   OFF
  yolo12n-baseline          yolo12n.pt         30     16   OFF
  yolo12m-baseline          yolo12m.pt         30      8   OFF
  yolov26n-augmented        yolo26n.pt         50     16   ON
  yolov26m-augmented        yolo26m.pt         50      8   ON
  yolo12n-augmented         yolo12n.pt         50     16   ON
  yolo12m-augmented         yolo12m.pt         50      8   ON


In [6]:
print(RUNS)

[{'run_name': 'yolov26n-baseline', 'model': 'yolo26n.pt', 'epochs': 30, 'batch': 16, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolov26m-baseline', 'model': 'yolo26m.pt', 'epochs': 30, 'batch': 8, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolo12n-baseline', 'model': 'yolo12n.pt', 'epochs': 30, 'batch': 16, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolo12m-baseline', 'model': 'yolo12m.pt', 'epochs': 30, 'batch': 8, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolov26n-augmented', 'model': 'yolo26n.pt', 'epochs': 50, 'batch': 16, 'hsv_h': 0

## 3. Training Loop

Each config gets its own MLflow run. Parameters, metrics, and the best weights are all logged automatically.

In [7]:
FIXED_KEYS = {'run_name', 'model', 'epochs', 'batch'}

for cfg in RUNS:
    run_name = cfg['run_name']
    aug_args = {k: v for k, v in cfg.items() if k not in FIXED_KEYS and v is not None}

    print(f'\n{"="*60}')
    print(f'Starting run: {run_name}')
    print(f'{"="*60}')

    mlflow.end_run()
    with mlflow.start_run(run_name=run_name):

        # ── Log all parameters
        mlflow.log_params({
            'model' : cfg['model'],
            'epochs': cfg['epochs'],
            'batch' : cfg['batch'],
            **SHARED,
            **aug_args,
        })

        # ── Train
        model   = YOLO(cfg['model'])
        results = model.train(
            data     = str(DATA_YAML),
            epochs   = cfg['epochs'],
            batch    = cfg['batch'],
            project  = 'runs/train',
            name     = run_name,
            exist_ok = True,
            verbose  = False,
            **SHARED,
            **aug_args,
        )

        # ── Log final metrics
        metrics = results.results_dict
        mlflow.log_metrics({
            'mAP50'    : metrics.get('metrics/mAP50(B)',     0),
            'mAP50-95' : metrics.get('metrics/mAP50-95(B)', 0),
            'precision': metrics.get('metrics/precision(B)', 0),
            'recall'   : metrics.get('metrics/recall(B)',    0),
            'box_loss' : metrics.get('val/box_loss',         0),
            'cls_loss' : metrics.get('val/cls_loss',         0),
            'fitness'  : results.fitness
        })

        # ── Per-epoch loss curves
        csv_path = Path(f'/content/runs/detect/runs/{run_name}/results.csv')
        if csv_path.exists():
            df_results = pd.read_csv(csv_path)
            df_results.columns = df_results.columns.str.strip()
            for _, row in df_results.iterrows():
                epoch = int(row['epoch'])
                mlflow.log_metrics({
                    'train/box_loss': row.get('train/box_loss', 0),
                    'train/cls_loss': row.get('train/cls_loss', 0),
                    'train/dfl_loss': row.get('train/dfl_loss', 0),
                    'val/box_loss'  : row.get('val/box_loss',   0),
                    'val/cls_loss'  : row.get('val/cls_loss',   0),
                    'val/dfl_loss'  : row.get('val/dfl_loss',   0),
                }, step=epoch)

        # ── Log best weights as artifact copy the best to mlflow storage
        best_weights = Path(f'/content/runs/detect/runs/train/{run_name}/weights/best.pt')
        if best_weights.exists():
            mlflow.log_artifact(str(best_weights), artifact_path='weights')

        print(f'Run complete — mAP50: {metrics.get("metrics/mAP50(B)", 0):.4f}')


Starting run: yolov26n-baseline
Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/HairNet-Compliance-Detection-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.0, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=yolov26n-baseline, nbs=64, nms=False, opset=None, op

2026/05/13 17:35:48 INFO mlflow.tracking.fluent: Experiment with name 'runs/train' does not exist. Creating a new experiment.
2026/05/13 17:35:48 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/05/13 17:35:48 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(d26534032ec14a2092b94310e055b8f0) to mlruns
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri mlruns'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /content/runs/detect/runs/train/yolov26n-baseline
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
: 0% ──────────── 0/18  12.1s


KeyboardInterrupt: 